# Pricing Financial Options with QMCJu

Translated from QMCJu's pricing_options.ipynb

Demonstrates European, Asian, Lookback, and Digital option pricing
using IID Monte Carlo and QMC methods.

In [1]:
using QMCJu
using Statistics
using Printf

## Parameters

In [2]:
S0 = 120.0            # initial stock price
r = 0.02              # risk-free interest rate
σ = 0.5               # volatility
K = 130.0             # strike price
T = 1.0               # time to maturity (years)
d = 52                # number of monitoring dates (weekly)

println("Option Pricing Parameters:")
println("  S₀ = $S0, K = $K, σ = $σ, r = $r, T = $T, d = $d")
println()

Option Pricing Parameters:
  S₀ = 120.0, K = 130.0, σ = 0.5, r = 0.02, T = 1.0, d = 52



European Options

In [3]:
println("="^60)
println("European Options")
println("="^60)

European Options


European Call — IID Monte Carlo

In [4]:
dd = IIDStdUniform(d; seed=7)
tm = BrownianMotion(dd)
f_eu_call = FinancialOption(tm;
    option_type=:european, call_put=:call,
    volatility=σ, start_price=S0, strike_price=K, interest_rate=r)
sc = CubMCCLT(f_eu_call; abs_tol=5.0, n_init=4096)
result = integrate(sc)
@printf("  European Call (IID MC):   %.4f  (n = %d)\n", result.solution, result.data[:n])

  European Call (IID MC):   20.5295  (n = 8192)


European Put — IID Monte Carlo

In [5]:
dd = IIDStdUniform(d; seed=7)
tm = BrownianMotion(dd)
f_eu_put = FinancialOption(tm;
    option_type=:european, call_put=:put,
    volatility=σ, start_price=S0, strike_price=K, interest_rate=r)
sc = CubMCCLT(f_eu_put; abs_tol=5.0, n_init=4096)
result = integrate(sc)
@printf("  European Put  (IID MC):   %.4f  (n = %d)\n", result.solution, result.data[:n])
println()

  European Put  (IID MC):   28.1176  (n = 8192)



Asian Options

In [6]:
println("="^60)
println("Asian Options (Arithmetic Mean)")
println("="^60)

Asian Options (Arithmetic Mean)


Asian Call — IID Monte Carlo

In [7]:
dd = IIDStdUniform(d; seed=7)
tm = BrownianMotion(dd)
f_as_call = FinancialOption(tm;
    option_type=:asian, call_put=:call, mean_type=:arithmetic,
    volatility=σ, start_price=S0, strike_price=K, interest_rate=r)
sc = CubMCCLT(f_as_call; abs_tol=0.05)
result = integrate(sc)
@printf("  Asian Call (IID MC):      %.4f  (n = %d)\n", result.solution, result.data[:n])

  Asian Call (IID MC):      10.5225  (n = 2218724)


Asian Call — Lattice QMC

In [8]:
dd = Lattice(d; randomize=true, seed=7)
tm = BrownianMotion(dd)
f_as_call_lat = FinancialOption(tm;
    option_type=:asian, call_put=:call, mean_type=:arithmetic,
    volatility=σ, start_price=S0, strike_price=K, interest_rate=r)
sc = CubQMCLatticeG(f_as_call_lat; abs_tol=0.05, n_init=2^8, n_reps=16)
result = integrate(sc)
@printf("  Asian Call (Lattice QMC): %.4f  (n/rep = %d)\n", result.solution, result.data[:n])

  Asian Call (Lattice QMC): 10.5318  (n/rep = 2048)


Asian Call — Digital Net QMC

In [9]:
dd = DigitalNetB2(d; randomize="LMS_DS", seed=7)
tm = BrownianMotion(dd)
f_as_call_dn = FinancialOption(tm;
    option_type=:asian, call_put=:call, mean_type=:arithmetic,
    volatility=σ, start_price=S0, strike_price=K, interest_rate=r)
sc = CubQMCNetG(f_as_call_dn; abs_tol=0.05, n_init=2^8, n_reps=16)
result = integrate(sc)
@printf("  Asian Call (Sobol' QMC):  %.4f  (n/rep = %d)\n", result.solution, result.data[:n])

  Asian Call (Sobol' QMC):  10.5388  (n/rep = 2048)


Asian Call — Geometric Mean

In [10]:
dd = IIDStdUniform(d; seed=7)
tm = BrownianMotion(dd)
f_as_geom = FinancialOption(tm;
    option_type=:asian, call_put=:call, mean_type=:geometric,
    volatility=σ, start_price=S0, strike_price=K, interest_rate=r)
sc = CubMCCLT(f_as_geom; abs_tol=0.05)
result = integrate(sc)
@printf("  Asian Call (Geometric):   %.4f  (n = %d)\n", result.solution, result.data[:n])
println()

  Asian Call (Geometric):   9.3407  (n = 1741615)



Lookback Options

In [11]:
println("="^60)
println("Lookback Options")
println("="^60)

dd = IIDStdUniform(d; seed=7)
tm = BrownianMotion(dd)
f_lb = FinancialOption(tm;
    option_type=:lookback, call_put=:call,
    volatility=σ, start_price=S0, strike_price=K, interest_rate=r)
sc = CubMCCLT(f_lb; abs_tol=0.5)
result = integrate(sc)
@printf("  Lookback Call (IID MC):   %.4f  (n = %d)\n", result.solution, result.data[:n])

dd = IIDStdUniform(d; seed=7)
tm = BrownianMotion(dd)
f_lb_put = FinancialOption(tm;
    option_type=:lookback, call_put=:put,
    volatility=σ, start_price=S0, strike_price=K, interest_rate=r)
sc = CubMCCLT(f_lb_put; abs_tol=5.0)
result = integrate(sc)
@printf("  Lookback Put  (IID MC):   %.4f  (n = %d)\n", result.solution, result.data[:n])
println()

Lookback Options
  Lookback Call (IID MC):   41.7933  (n = 123393)
  Lookback Put  (IID MC):   45.1822  (n = 2048)



Digital Options

In [12]:
println("="^60)
println("Digital (Binary) Options")
println("="^60)

dd = IIDStdUniform(d; seed=7)
tm = BrownianMotion(dd)
f_dg = FinancialOption(tm;
    option_type=:digital, call_put=:call,
    volatility=σ, start_price=S0, strike_price=K, interest_rate=r)
sc = CubMCCLT(f_dg; abs_tol=0.01)
result = integrate(sc)
@printf("  Digital Call (IID MC):    %.4f  (n = %d)\n", result.solution, result.data[:n])
@printf("  (Probability that S(T) > K, discounted)\n")
println()

println("="^60)
println("All option pricing demos completed!")

Digital (Binary) Options
  Digital Call (IID MC):    0.3492  (n = 22204)
  (Probability that S(T) > K, discounted)

All option pricing demos completed!
